# EXP-003: 학습 설정 개선 (원본 19 피처 유지)

**EXP-002 결론:** 페어와이즈 조합 + TargetEncoder → 효과 없음 (오히려 악화)  
**방향 전환:** 피처는 그대로, 학습 설정 개선으로 성능 향상 시도

| 실험 | 변경점 |
|------|--------|
| A | 10-Fold CV (5 → 10) |
| B | Early Stopping + 더 많은 estimators (1000 → 5000) + 낮은 lr (0.05 → 0.02) |
| C | A + B 결합 |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings('ignore')

mpl.rcParams['font.family'] = 'Malgun Gothic'
mpl.rcParams['axes.unicode_minus'] = False

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

TARGET = 'Irrigation_Need'
CAT_COLS = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

target_le = LabelEncoder()
y = target_le.fit_transform(train[TARGET])

for col in CAT_COLS:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]])
    le.fit(combined)
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

drop_cols = ['id', TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]
X = train[feature_cols]
X_test = test[feature_cols]
sample_weights = compute_sample_weight('balanced', y)

print(f"피처: {len(feature_cols)}개 (원본 유지)")
print(f"Train: {X.shape}, Test: {X_test.shape}")

피처: 19개 (원본 유지)
Train: (630000, 19), Test: (270000, 19)


In [2]:
SEED = 42

def train_and_evaluate(X, y, X_test, sample_weights, params, n_folds=5, seed=42, early_stopping=None):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof_preds = np.zeros((len(X), 3))
    test_preds = np.zeros((len(X_test), 3))
    fold_scores = []
    best_iterations = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        sw_tr = sample_weights[train_idx]

        fit_params = dict(sample_weight=sw_tr, eval_set=[(X_val, y_val)], verbose=0)
        if early_stopping:
            params_copy = {**params, 'early_stopping_rounds': early_stopping}
            model = XGBClassifier(**params_copy)
        else:
            model = XGBClassifier(**params)

        model.fit(X_tr, y_tr, **fit_params)

        oof_preds[val_idx] = model.predict_proba(X_val)
        test_preds += model.predict_proba(X_test) / n_folds

        score = balanced_accuracy_score(y_val, oof_preds[val_idx].argmax(axis=1))
        fold_scores.append(score)
        best_iter = model.best_iteration if hasattr(model, 'best_iteration') and model.best_iteration else params.get('n_estimators', '?')
        best_iterations.append(best_iter)
        print(f"  Fold {fold}: {score:.5f} (best_iter: {best_iter})")

    oof_labels = oof_preds.argmax(axis=1)
    overall = balanced_accuracy_score(y, oof_labels)
    print(f"  Overall OOF: {overall:.5f} (Mean: {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f})")
    return oof_preds, test_preds, overall

## 실험 A: 10-Fold CV (기존 파라미터 동일)

In [3]:
params_baseline = {
    'n_estimators': 1000,
    'max_depth': 8,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': 0,
}

print("=== 실험 A: 10-Fold CV (기존 파라미터) ===")
oof_a, test_a, score_a = train_and_evaluate(
    X, y, X_test, sample_weights, params_baseline, n_folds=10, seed=SEED)

=== 실험 A: 10-Fold CV (기존 파라미터) ===
  Fold 0: 0.96504 (best_iter: 1000)
  Fold 1: 0.96667 (best_iter: 1000)
  Fold 2: 0.96622 (best_iter: 1000)
  Fold 3: 0.96833 (best_iter: 1000)
  Fold 4: 0.96674 (best_iter: 1000)
  Fold 5: 0.96801 (best_iter: 1000)
  Fold 6: 0.96773 (best_iter: 1000)
  Fold 7: 0.96434 (best_iter: 1000)
  Fold 8: 0.96698 (best_iter: 1000)
  Fold 9: 0.96598 (best_iter: 1000)
  Overall OOF: 0.96660 (Mean: 0.96660 ± 0.00120)


## 실험 B: Early Stopping + 낮은 LR + 많은 Estimators (5-Fold)

In [4]:
params_b = {
    'n_estimators': 5000,
    'max_depth': 6,            # 8 → 6 (과적합 방지)
    'learning_rate': 0.02,     # 0.05 → 0.02 (더 세밀하게)
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': 0,
}

print("=== 실험 B: Early Stopping + lr=0.02 + n_est=5000 (5-Fold) ===")
oof_b, test_b, score_b = train_and_evaluate(
    X, y, X_test, sample_weights, params_b, n_folds=5, seed=SEED, early_stopping=200)

=== 실험 B: Early Stopping + lr=0.02 + n_est=5000 (5-Fold) ===
  Fold 0: 0.96663 (best_iter: 4985)
  Fold 1: 0.96851 (best_iter: 4682)
  Fold 2: 0.96858 (best_iter: 4787)
  Fold 3: 0.96738 (best_iter: 4750)
  Fold 4: 0.96738 (best_iter: 4896)
  Overall OOF: 0.96770 (Mean: 0.96770 ± 0.00075)


## 실험 C: A + B 결합 (10-Fold + Early Stopping + 낮은 LR)

In [5]:
print("=== 실험 C: 10-Fold + Early Stopping + lr=0.02 + n_est=5000 ===")
oof_c, test_c, score_c = train_and_evaluate(
    X, y, X_test, sample_weights, params_b, n_folds=10, seed=SEED, early_stopping=200)

=== 실험 C: 10-Fold + Early Stopping + lr=0.02 + n_est=5000 ===
  Fold 0: 0.96727 (best_iter: 4993)
  Fold 1: 0.96811 (best_iter: 4907)
  Fold 2: 0.96760 (best_iter: 4975)
  Fold 3: 0.96954 (best_iter: 4985)
  Fold 4: 0.96864 (best_iter: 4992)
  Fold 5: 0.96895 (best_iter: 4991)
  Fold 6: 0.96928 (best_iter: 4991)
  Fold 7: 0.96621 (best_iter: 4963)
  Fold 8: 0.96798 (best_iter: 4999)
  Fold 9: 0.96767 (best_iter: 4986)
  Overall OOF: 0.96812 (Mean: 0.96812 ± 0.00096)


## 결과 비교 및 제출

In [6]:
baseline_oof = 0.96599

results = {
    'EXP-001 Baseline (5F, lr=0.05, 1000est)': baseline_oof,
    'A: 10-Fold (기존 파라미터)': score_a,
    'B: 5-Fold + ES + lr=0.02 + 5000est': score_b,
    'C: 10-Fold + ES + lr=0.02 + 5000est': score_c,
}

print("=" * 65)
print(f"{'실험':<42} {'OOF':>8} {'변화량':>10}")
print("=" * 65)
for name, score in results.items():
    diff = score - baseline_oof
    print(f"{name:<42} {score:>8.5f} {diff:>+10.5f}")
print("=" * 65)

best_name = max(results, key=results.get)
print(f"\n최고 성능: {best_name} ({results[best_name]:.5f})")

실험                                              OOF        변화량
EXP-001 Baseline (5F, lr=0.05, 1000est)     0.96599   +0.00000
A: 10-Fold (기존 파라미터)                        0.96660   +0.00061
B: 5-Fold + ES + lr=0.02 + 5000est          0.96770   +0.00171
C: 10-Fold + ES + lr=0.02 + 5000est         0.96812   +0.00213

최고 성능: C: 10-Fold + ES + lr=0.02 + 5000est (0.96812)


In [7]:
# 최고 성능 실험으로 제출 파일 생성
for tag, preds in [('a_10fold', test_a), ('b_es_lowlr', test_b), ('c_10fold_es', test_c)]:
    labels = target_le.inverse_transform(preds.argmax(axis=1))
    sub = pd.DataFrame({'id': test['id'], 'Irrigation_Need': labels})
    path = f'../submissions/submission_exp003_{tag}.csv'
    sub.to_csv(path, index=False)
    dist = sub['Irrigation_Need'].value_counts()
    print(f"{path}")
    print(f"  {dict(dist)}\n")

../submissions/submission_exp003_a_10fold.csv
  {'Low': np.int64(159932), 'Medium': np.int64(101235), 'High': np.int64(8833)}

../submissions/submission_exp003_b_es_lowlr.csv
  {'Low': np.int64(159881), 'Medium': np.int64(101213), 'High': np.int64(8906)}

../submissions/submission_exp003_c_10fold_es.csv
  {'Low': np.int64(159864), 'Medium': np.int64(101197), 'High': np.int64(8939)}



## 실험 D: n_estimators=10000 (10-Fold + ES + lr=0.02)

실험 C에서 best_iter가 4900~5000으로 수렴 안 됨 → estimators를 2배로 확장

In [9]:
params_d = {
    'n_estimators': 10000,     # 5000 → 10000
    'max_depth': 6,
    'learning_rate': 0.02,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cuda',          # GPU 사용
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': 0,
}

print("=== 실험 D: 10-Fold + ES(300) + lr=0.02 + n_est=10000 ===")
oof_d, test_d, score_d = train_and_evaluate(
    X, y, X_test, sample_weights, params_d, n_folds=10, seed=SEED, early_stopping=300)

=== 실험 D: 10-Fold + ES(300) + lr=0.02 + n_est=10000 ===
  Fold 0: 0.96672 (best_iter: 5490)
  Fold 1: 0.96812 (best_iter: 5065)
  Fold 2: 0.96728 (best_iter: 4993)
  Fold 3: 0.96996 (best_iter: 5378)
  Fold 4: 0.96902 (best_iter: 5369)
  Fold 5: 0.96899 (best_iter: 5425)
  Fold 6: 0.96906 (best_iter: 5186)
  Fold 7: 0.96619 (best_iter: 5080)
  Fold 8: 0.96854 (best_iter: 5270)
  Fold 9: 0.96702 (best_iter: 5230)
  Overall OOF: 0.96809 (Mean: 0.96809 ± 0.00117)


In [10]:
# 전체 결과 비교
results_all = {
    'EXP-001 Baseline (5F, lr=0.05, 1000est)': 0.96599,
    'A: 10-Fold (기존 파라미터)': score_a,
    'B: 5-Fold + ES + lr=0.02 + 5000est': score_b,
    'C: 10-Fold + ES + lr=0.02 + 5000est': score_c,
    'D: 10-Fold + ES(300) + lr=0.02 + 10000est': score_d,
}

print("=" * 65)
print(f"{'실험':<45} {'OOF':>8} {'변화량':>10}")
print("=" * 65)
for name, score in results_all.items():
    diff = score - 0.96599
    print(f"{name:<45} {score:>8.5f} {diff:>+10.5f}")
print("=" * 65)

best = max(results_all, key=results_all.get)
print(f"\n최고 성능: {best} ({results_all[best]:.5f})")

# D 제출 파일 생성
labels = target_le.inverse_transform(test_d.argmax(axis=1))
sub = pd.DataFrame({'id': test['id'], 'Irrigation_Need': labels})
sub.to_csv('../submissions/submission_exp003_d_10k.csv', index=False)
print(f"\n저장: submission_exp003_d_10k.csv")
print(f"분포: {dict(sub['Irrigation_Need'].value_counts())}")

실험                                                 OOF        변화량
EXP-001 Baseline (5F, lr=0.05, 1000est)        0.96599   +0.00000
A: 10-Fold (기존 파라미터)                           0.96660   +0.00061
B: 5-Fold + ES + lr=0.02 + 5000est             0.96770   +0.00171
C: 10-Fold + ES + lr=0.02 + 5000est            0.96812   +0.00213
D: 10-Fold + ES(300) + lr=0.02 + 10000est      0.96809   +0.00210

최고 성능: C: 10-Fold + ES + lr=0.02 + 5000est (0.96812)

저장: submission_exp003_d_10k.csv
분포: {'Low': np.int64(159869), 'Medium': np.int64(101199), 'High': np.int64(8932)}
